# Test XGBoost

Load the generated XGBoost checkpoint and run inference through the same preprocessing path.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[4]
PIPELINE_ROOT = REPO_ROOT / "ai-ml" / "training_pipeline"
XGB_ROOT = REPO_ROOT / "ai-ml" / "models" / "core-models" / "xgb"
REPORT_PATH = XGB_ROOT / "reports" / "trey_xgb_core_v2_experiment_summary.json"
CONFIG_PATH = XGB_ROOT / "configs" / "xgboost_config.yaml"

sys.path.insert(0, str(PIPELINE_ROOT))
from src.core.checkpoint_manager import CheckpointManager
from src.core.pipeline import _load_preprocessing_runtime_config, _normalize_selected_features
from src.data.dataset_loader import DatasetLoader
from src.preprocessing.preprocess import preprocess_features
from src.utils.config_loader import load_config

summary = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
checkpoint_path = Path(summary["checkpoint_path"])
if not checkpoint_path.is_absolute():
    checkpoint_path = REPO_ROOT / checkpoint_path

model = CheckpointManager(checkpoint_path.parent).load(checkpoint_path.name)
config = load_config(CONFIG_PATH)
dataset_path = REPO_ROOT / config["dataset"]["path"]
df = pd.read_csv(dataset_path)
cleaning_config, preprocessing_config = _load_preprocessing_runtime_config(REPO_ROOT, config, None)
processed_df, _ = preprocess_features(
    data=df,
    cleaning_config=cleaning_config,
    preprocessing_config=preprocessing_config,
    selected_features=_normalize_selected_features(preprocessing_config.get("selected_features")),
    target_column=config["dataset"]["target_column"],
)
x, _ = DatasetLoader.separate_features_and_target(processed_df, config["dataset"]["target_column"])
predictions = model.predict(x.head(10))
print("checkpoint:", checkpoint_path)
print("predictions:", predictions.tolist())

checkpoint: C:\Users\asadd\OneDrive\Desktop\Phoenix\Phoenix\ai-ml\models\core-models\xgb\checkpoints\trey_xgb_core_v2\core_xgb_xgboost_trey_xgb_core_v2_best.joblib
predictions: [2, 2, 1, 0, 0, 1, 1, 1, 1, 3]
